In [ ]:
# Find the pipeline package without walking the DICOM tree.
import sys, os, time, gc, json
from pathlib import Path

T0 = time.time()
def log(msg):
    print(f"[{time.time() - T0:7.1f}s] {msg}", flush=True)

def walk(root="/kaggle/input", max_depth=8, suffixes=None):
    base = len(Path(root).parts)
    for dirpath, dirnames, filenames in os.walk(root):
        here = Path(dirpath)
        if len(here.parts) - base >= max_depth:
            dirnames[:] = []
        else:
            dirnames[:] = [d for d in dirnames if not d.lower().endswith(("_series", "_images"))]
        for f in filenames:
            if suffixes is None or Path(f).suffix.lower() in suffixes:
                yield here / f

PKG = next((p.parent.parent for p in walk(suffixes=(".py",))
            if p.name == "targets.py" and p.parent.name == "rsna_knee"), None)
if PKG is None:
    raise FileNotFoundError("attach the rsna-knee-pipeline dataset (v2, it must contain rsna_knee/targets.py)")
sys.path.insert(0, str(PKG))

import numpy as np
import pandas as pd
from rsna_knee.report_labels import TARGETS
log(f"pipeline: {PKG}")


In [ ]:
# CPU notebook (Accelerator: None). Reads only DICOM headers of every training series and
# writes slice_audit.csv: how the largest-spread axis order compares with the stack-normal order,
# the slice spacing, pixel anisotropy and the physical span of one 3-slice cache window.
from concurrent.futures import ThreadPoolExecutor
import pydicom
from rsna_knee.infer import find_root
from rsna_knee.cache3 import natural_key

ROOT = find_root()
series = pd.read_csv(ROOT / "train_series.csv", dtype=str)
PLANE_AXIS = {"Sagittal": 0, "Coronal": 1, "Axial": 2}

def audit(row):
    d = ROOT / "train_series" / row.StudyInstanceUID / row.SeriesInstanceUID
    out = {"StudyInstanceUID": row.StudyInstanceUID, "SeriesInstanceUID": row.SeriesInstanceUID,
           "plane": row.Anatomical_Plane}
    try:
        names = sorted((e.name for e in os.scandir(d) if e.name.endswith(".dcm")), key=natural_key)
    except Exception:
        return out
    pos, iop, ps, th, sb, lat, pid = [], None, None, None, None, None, None
    for n in names:
        try:
            h = pydicom.dcmread(d / n, stop_before_pixels=True, force=True)
        except Exception:
            continue
        p = getattr(h, "ImagePositionPatient", None)
        if p is not None and len(p) >= 3:
            pos.append([float(v) for v in p[:3]])
        if iop is None and getattr(h, "ImageOrientationPatient", None) is not None:
            iop = np.array([float(v) for v in h.ImageOrientationPatient[:6]])
        if ps is None and getattr(h, "PixelSpacing", None) is not None:
            ps = [float(v) for v in h.PixelSpacing[:2]]
        th = th or (float(h.SliceThickness) if getattr(h, "SliceThickness", None) else None)
        sb = sb or (float(h.SpacingBetweenSlices) if getattr(h, "SpacingBetweenSlices", None) else None)
        lat = lat or str(getattr(h, "Laterality", "") or getattr(h, "ImageLaterality", "") or "") or None
        pid = pid or str(getattr(h, "PatientID", "") or "") or None
    out.update(n_files=len(names), n_pos=len(pos), has_iop=iop is not None, thickness=th, spacing_tag=sb,
               laterality_tag=lat, patient_id=pid)
    if ps:
        out["px_row"], out["px_col"] = ps
    if len(pos) >= 3 and iop is not None:
        a = np.array(pos)
        axis = int(np.argmax(a.max(0) - a.min(0)))
        nrm = np.cross(iop[:3], iop[3:])
        k_axis, k_nrm = np.argsort(a[:, axis], kind="stable"), np.argsort(a @ nrm, kind="stable")
        out["axis"] = axis
        out["axis_is_plane_axis"] = axis == PLANE_AXIS.get(row.Anatomical_Plane, -1)
        out["same_order"] = bool((k_axis == k_nrm).all())
        out["reversed_order"] = bool((k_axis == k_nrm[::-1]).all())
        proj = np.sort(a @ nrm)
        gaps = np.diff(proj)
        out["spacing_ipp"] = float(np.median(np.abs(gaps))) if len(gaps) else None
        out["max_tilt_deg"] = float(np.degrees(np.arccos(np.clip(np.abs(nrm).max() / np.linalg.norm(nrm), 0, 1))))
        n = len(pos)
        lo, hi = round(0.06 * (n - 1)), round(0.94 * (n - 1))
        out["window_span_mm"] = float(2 * (hi - lo) / 15 * out["spacing_ipp"])
        out["window_acquired_slices"] = float(2 * (hi - lo) / 15 + 1)
        out["iop_plane"] = ["Sagittal", "Coronal", "Axial"][int(np.argmax(np.abs(nrm)))]
        out["median_x"] = float(np.median(a[:, 0]))
    return out

with ThreadPoolExecutor(32) as ex:
    rows = list(ex.map(audit, series.itertuples(index=False)))
aud = pd.DataFrame(rows)
aud.to_csv("/kaggle/working/slice_audit.csv", index=False)
ok = aud.dropna(subset=["same_order"])
log(f"series {len(aud)}, with geometry {len(ok)}")
log(f"axis order equals stack-normal order: {(ok.same_order | ok.reversed_order).mean():.4f} "
    f"(same direction {ok.same_order.mean():.4f}, reversed {ok.reversed_order.mean():.4f})")
log(f"largest-spread axis is the plane's own axis: {ok.axis_is_plane_axis.mean():.4f}")
log(f"pixel spacing anisotropic (>1%): {((aud.px_row - aud.px_col).abs() / aud.px_row > 0.01).mean():.4f}")
log("window span (mm) by plane:\n" + ok.groupby("plane").window_span_mm.describe().round(2).to_string())
log("tilt from the nearest axis (deg) by plane:\n" + ok.groupby("plane").max_tilt_deg.describe().round(1).to_string())
log(f"plane from the slice normal agrees with the series label: {(ok.iop_plane == ok.plane).mean():.4f}")
log(f"series without enough positions or orientation: {1 - len(ok) / len(aud):.4f}")
per_study = aud.dropna(subset=["patient_id"]).groupby("StudyInstanceUID").patient_id.first()
shared = per_study.map(per_study.value_counts()) > 1
log(f"PatientID present for {len(per_study)} of {aud.StudyInstanceUID.nunique()} studies; "
    f"{per_study.nunique()} distinct values; studies sharing a PatientID with another study: {int(shared.sum())}")
FOLDS_CSV = next((p for p in walk(suffixes=(".csv",)) if p.name == "train_folds.csv"), None)
if FOLDS_CSV is not None:
    folds = pd.read_csv(FOLDS_CSV).set_index("StudyInstanceUID").fold
    cross = pd.DataFrame({"pid": per_study, "fold": folds.reindex(per_study.index)}).groupby("pid").fold.nunique()
    log(f"PatientIDs whose studies fall into more than one fold: {int((cross > 1).sum())}")
lt = aud.laterality_tag.notna()
log(f"laterality tag present: {lt.mean():.4f}; median x at least 20 mm from 0 where absent: "
    f"{(aud.loc[~lt, 'median_x'].abs() >= 20).mean():.4f}")
